In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
Path("./images").mkdir(exist_ok=True)

# Main table

In [ ]:
df = pd.concat([
    pd.read_csv(f"../results/main_table/{dataset}/main_table_summary.csv")
    for dataset in ["20ng", "ag_news",]# "dbpedia14"]
])
df = df.drop(["accuracy_mean", "accuracy_std", "seed_mean", "seed_std"], axis=1)
df["dataset"] = df.dataset.replace({
    "20ng": "20NG",
    "ag_news": "AG News",
    "dbpedia14": "DBpedia14",
})
df["model"] = df.model.replace({
    "AttentiveTopicModel": "AARTM",
    "AttentiveTopicModelNoNWT": "AARTM-no-N",
    "CombinedTM": "CTM",
    "ContextualTop2Vec": "C-Top2Vec"
})
df = df.astype({
    col: float for col in df.columns if col not in ["dataset", "model"]
})
df = df.reset_index(drop=True)
df

In [ ]:
metrics = [
    ("npmi_10", "NPMI@10"),
    ("c_v_10", "C_v@10"),
    ("topic_diversity_25", "TD@25"),
    ("macro_f1", "Macro-F1"),
    ("train_time_sec", "Time"),
]

long_rows = []
for _, row in df.iterrows():
    for metric, metric_name in metrics:
        mean_col = f"{metric}_mean"
        std_col = f"{metric}_std"

        long_rows.append({
            "Dataset": row["dataset"],
            "Model": row["model"],
            "Metric": metric_name,
            "Mean": row[mean_col],
            "Std": row[std_col],
        })

plot_df = pd.DataFrame(long_rows)

sns.set_theme(
    style="whitegrid",
    context="paper",
    font_scale=1.0,
)

models = [
    "AARTM", "AARTM-no-N", "LDA", "NMF",
    "BERTopic", "BigARTM", "C-Top2Vec",
    "CTM", "BTM",
]
datasets = ["20NG", "AG News"]
metrics = ["NPMI@10", "TD@25", "C_v@10", "Time"]

palette = {
    "AARTM": "#50a38a",
    "AARTM-no-N": "#a5f0d8",
    "LDA": "#938fd1",
    "NMF": "#e27cb1",
    "BERTopic": "#dd9e6e",
    "BigARTM": "#797BDA",
    "C-Top2Vec": "#8FC58C",
    "CTM": "#B890D8",
    "BTM": "#CE9578",
}

n_rows = len(datasets)
n_cols = len(metrics)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(2.4 * n_cols, 1.9 * n_rows),
    sharex=False,
    sharey=False,
    constrained_layout=False,
)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)
if n_cols == 1:
    axes = np.expand_dims(axes, axis=1)

x = np.arange(len(models))
bar_width = 0.72

for i, dataset in enumerate(datasets):
    for j, metric in enumerate(metrics):
        ax = axes[i, j]

        sub = plot_df[
            (plot_df["Dataset"] == dataset) &
            (plot_df["Metric"] == metric)
        ].copy()

        sub["Model"] = pd.Categorical(
            sub["Model"],
            categories=models,
            ordered=True,
        )
        sub = sub.sort_values("Model")

        means = sub["Mean"].values
        stds = sub["Std"].values
        colors = [palette[m] for m in sub["Model"].astype(str).values]

        ax.bar(
            x,
            means,
            yerr=stds,
            width=bar_width,
            color=colors,
            edgecolor="black",
            linewidth=0.4,
            error_kw={
                "elinewidth": 0.8,
                "capsize": 2,
                "capthick": 0.8,
            },
        )

        if metric == "NPMI@10":
            ax.axhline(0.0, color="black", linewidth=0.6)

        if i == 0:
            ax.set_title(metric, fontsize=10, pad=6)

        if j == 0:
            ax.set_ylabel(dataset, fontsize=10, fontweight="bold")
        else:
            ax.set_ylabel("")

        ax.set_xticks(x)
        if i == n_rows - 1:
            ax.set_xticklabels(models, rotation=45, ha="right", fontsize=8)
        else:
            ax.set_xticklabels([])

        ax.tick_params(axis="y", labelsize=8)
        ax.tick_params(axis="x", length=0)

        if metric == "TD@25":
            ax.set_ylim(0.3, 1.0)
        # elif metric == "C_v@10":
        #     ax.set_ylim(0.4, 0.8)
        elif metric == "Macro-F1":
            ax.set_ylim(0.85, 0.9)
        # elif metric == "NPMI@10":
        #     ax.set_ylim(0.0, 0.4)
        elif metric == "Time":
            # ax.set_ylim(0, max(means + stds) * 1.15)
            ax.set_yscale("log")
            pass

        sns.despine(ax=ax)

handles = [
    plt.Rectangle(
        (0, 0),
        1,
        1,
        color=palette[m],
        ec="black",
        linewidth=0.4,
        label=m,
    )
    for m in models
]

fig.legend(
    handles=handles,
    labels=models,
    loc="lower center",
    ncol=len(models),
    frameon=False,
    bbox_to_anchor=(0.5, -0.02),
    fontsize=9,
)

# fig.suptitle(
#     "Main comparison across datasets and metrics",
#     fontsize=12,
#     y=1.02,
# )
plt.tight_layout(rect=[0, 0.08, 1, 0.98])

plt.savefig("images/table2_main_table.pdf", bbox_inches="tight")
plt.savefig("images/table2_main_table.png", bbox_inches="tight", dpi=300)
plt.show()

# Effective context ablation

In [ ]:
df = pd.concat([
    pd.read_csv(f"../results/ablation/context_length/{dataset}/ablation_summary.csv")
    for dataset in ["20ng", "ag_news", "dbpedia14"]
])
df["C"] = df["ctx_len"].astype(int)
df = df.drop([
    "model_variant", "self_aware_context", "num_attn_passes", "decorrelation_tau",
    "accuracy_mean", "accuracy_std", "seed_mean", "seed_std", "ctx_len",
], axis=1)
df["dataset"] = df.dataset.replace({
    "20ng": "20NG",
    "ag_news": "AG News",
    "dbpedia14": "DBpedia14",
})
df = df.astype({
    col: float for col in df.columns if col not in ["dataset", "C"]
})
df = df.reset_index(drop=True)

df["normalized_perplexity_mean"] = (
    df.groupby("dataset")["perplexity_test_mean"]
           .transform(lambda x: x / x.max())
)

df["inverse_gamma"] = 1.0 / df["gamma"]
df["effective_radius"] = np.minimum(df["C"], df["inverse_gamma"])

df

In [ ]:
metrics = {
    "npmi_10_mean": "NPMI@10",
    "c_v_10_mean": "C_v@10",
    "normalized_perplexity_mean": "Normalized perplexity",
}
metrics_cols = list(metrics.keys())

agg = (
    df.groupby(["dataset", "effective_radius"], as_index=False)[metrics_cols]
      .mean()
)

plot_df = agg.melt(
    id_vars=["dataset", "effective_radius"],
    value_vars=metrics_cols,
    var_name="metric",
    value_name="value"
)
plot_df["metric"] = plot_df.metric.map(metrics)

sns.set_theme(style="whitegrid", context="paper", font_scale=1.4)

g = sns.relplot(
    data=plot_df,
    x="effective_radius",
    y="value",
    hue="dataset",
    col="metric",
    col_wrap=1,
    kind="line",
    marker="o",
    facet_kws={"sharey": False, "sharex": True},
    height=3.0,
    aspect=1.15,
    linewidth=3.0,
)

for i, ax in enumerate(g.axes.flat):
    ax.set_xscale("log")
    ax.grid(True, which="both", linestyle="--", alpha=0.35)

g.set_titles("{col_name}")
g.set_ylabels("")
g.axes.flat[-1].set_xlabel(r"Effective context radius $1/\gamma$")

g._legend.set_title("Dataset")
g._legend.set_bbox_to_anchor([0.98, 0.15])

plt.tight_layout()
plt.savefig("images/table3_context_length.pdf", bbox_inches="tight")
plt.savefig("images/table3_context_length.png", bbox_inches="tight", dpi=300)
plt.show()

# Self-aware context ablation

In [ ]:
df = pd.concat([
    pd.read_csv(f"../results/ablation/self_aware/{dataset}/ablation_summary.csv")
    for dataset in ["20ng", "ag_news", "dbpedia14"]
])
df["C"] = df["ctx_len"].astype(int)
df = df.drop([
    "model_variant", "num_attn_passes", "decorrelation_tau",
    "accuracy_mean", "accuracy_std", "seed_mean", "seed_std", "ctx_len",
], axis=1)
df["dataset"] = df.dataset.replace({
    "20ng": "20NG",
    "ag_news": "AG News",
    "dbpedia14": "DBpedia14",
})
df = df.astype({
    col: float for col in df.columns if col not in ["dataset", "C", "self_aware_context"]
})
df = df.reset_index(drop=True)

df

In [ ]:
settings_to_plot = [
    # (10, 0.1),
    (100, 0.01),
]

metrics = [
    ("macro_f1", "Macro-F1"),
    ("c_v_10", "C_v@10"),
    ("topic_diversity_25", "TD@25"),
    ("perplexity_test", "Perplexity"),
]

datasets = ["20NG", "AG News", "DBpedia14"]

label_map = {
    False: "Self-excluding",
    True: "Self-aware",
}

colors = {
    False: "#4C72B0",
    True: "#DD8452",
}

sns.set_theme(style="whitegrid", context="paper", font_scale=1.8)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(7, 5 * n_rows),
    sharex=True,
)

bar_width = 0.36
x = np.arange(len(datasets))

for row_idx, (C, gamma) in enumerate(settings_to_plot):
    sub = df[(df["C"] == C) & (df["gamma"] == gamma)]

    for col_idx, (metric_key, metric_label) in enumerate(metrics):
        # row_idx, col_idx = divmod(col_idx, 2)
        ax = axes[*divmod(col_idx, 2)]

        for j, self_aware in enumerate([False, True]):
            vals = []
            errs = []

            for dataset in datasets:
                item = sub[
                    (sub["dataset"] == dataset)
                    & (sub["self_aware_context"] == self_aware)
                ]
                if len(item) != 1:
                    raise ValueError(
                        f"Missing or duplicated row for "
                        f"dataset={dataset}, C={C}, gamma={gamma}, "
                        f"self_aware={self_aware}"
                    )

                vals.append(item[f"{metric_key}_mean"].iloc[0])
                errs.append(item[f"{metric_key}_std"].iloc[0])

            offset = (j - 0.5) * bar_width

            ax.bar(
                x + offset,
                vals,
                width=bar_width,
                yerr=errs,
                capsize=3,
                label=label_map[self_aware],
                color=colors[self_aware],
                edgecolor="black",
                linewidth=0.4,
            )

        ax.set_xticks(x)
        ax.set_xticklabels(datasets, rotation=30, ha="right")
        ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.5)

        if row_idx == 0:
            ax.set_title(metric_label)

        if metrics[col_idx][0] == "macro_f1":
            ax.set_ylim([0.5, 1.05])
        elif metrics[col_idx][0] == "c_v_10":
            ax.set_ylim([0.3, 0.8])
        elif metrics[col_idx][0] == "topic_diversity_25":
            ax.set_ylim([0.5, 1.05])
        elif metrics[col_idx][0] == "perplexity_test":
            ax.set_yscale("log")
        else:
            raise ValueError(f"Set ylim for a new metric: {metrics[col_idx][0]}")

handles = [
    plt.Rectangle(
        (0, 0),
        1,
        1,
        color=colors[m],
        ec="black",
        linewidth=0.4,
        label=m,
    )
    for m in [False, True]
]

fig.legend(
    handles=handles,
    labels=[label_map[m] for m in [False, True]],
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.55, -0.06),
)

fig.tight_layout()

plt.savefig("images/table4_self_aware_context.pdf", bbox_inches="tight")
plt.savefig("images/table4_self_aware_context.png", bbox_inches="tight", dpi=300)
plt.show()

# Attention passes ablation

In [ ]:
df = pd.concat([
    pd.read_csv(f"../results/ablation/attn_passes/{dataset}/ablation_summary.csv")
    for dataset in ["20ng", "ag_news", "dbpedia14"]
])
df = df[df["gamma"] != 0.5]
df["C"] = df["ctx_len"].astype(int)
df["L"] = df["num_attn_passes"].astype(int)
df = df.drop([
    "model_variant", "num_attn_passes", "decorrelation_tau", "gamma",
    "accuracy_mean", "accuracy_std", "seed_mean", "seed_std", "ctx_len",
    "self_aware_context",
], axis=1)
df["dataset"] = df.dataset.replace({
    "20ng": "20NG",
    "ag_news": "AG News",
    "dbpedia14": "DBpedia14",
})
df = df.astype({
    col: float for col in df.columns if col not in ["dataset", "C", "L"]
})
df = df.reset_index(drop=True)

df

In [ ]:
from matplotlib.ticker import MaxNLocator, FormatStrFormatter


datasets=["20NG", "AG News", "DBpedia14"]
context_values=[1, 2, 3, 4, 5]
metrics = [
    ("macro_f1_mean", "macro_f1_std", "Macro-F1"),
    ("topic_diversity_25_mean", "topic_diversity_25_std", "TD@25"),
    ("npmi_10_mean", "npmi_10_std", "NPMI@10"),
    ("c_v_10_mean", "c_v_10_std", "C_v@10"),
]

plot_df = df.sort_values(["dataset", "C", "L"])

sns.set_theme(style="whitegrid", context="paper", font_scale=2.2)

fig, axes = plt.subplots(
    len(datasets),
    len(metrics),
    figsize=(11, 7),
    sharex=True,
    constrained_layout=True,
)

if len(datasets) == 1:
    axes = axes[None, :]

if len(metrics) == 1:
    axes = axes[:, None]

all_contexts = sorted(plot_df["C"].unique())
palette = dict(zip(all_contexts, sns.color_palette("tab10", len(all_contexts))))

for row_idx, dataset in enumerate(datasets):
    sub_dataset = plot_df[plot_df["dataset"] == dataset]

    for col_idx, (mean_col, std_col, metric_name) in enumerate(metrics):
        ax = axes[row_idx, col_idx]

        for C, group in sub_dataset.groupby("C"):
            group = group.sort_values("L")

            x = group["L"].to_numpy()
            y = group[mean_col].to_numpy()
            yerr = group[std_col].to_numpy()

            ax.plot(
                x,
                y,
                marker="o",
                linewidth=1.8,
                markersize=3.5,
                color=palette[C],
                label=f"$C={C}$",
            )

            ax.fill_between(
                x,
                y - yerr,
                y + yerr,
                color=palette[C],
                alpha=0.15,
                linewidth=0,
            )

        if row_idx == 0:
            ax.set_title(metric_name)

        if col_idx == 0:
            ax.set_ylabel(dataset)

        if row_idx == len(datasets) - 1:
            ax.set_xlabel("$L$")

        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

        # Rounded tick labels
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))

        ax.grid(True, alpha=0.35)

handles, labels = axes[0, -1].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="Context radius",
    loc="lower center",
    bbox_to_anchor=(0.53, -0.15),
    ncol=min(len(labels), 5),
    frameon=True,
)

plt.savefig("images/table5_attn_passes.pdf", bbox_inches="tight")
plt.savefig("images/table5_attn_passes.png", bbox_inches="tight", dpi=300)
plt.show()

# No N_wt

In [ ]:
df_no_ntw = pd.concat([
    pd.read_csv(f"../results/ablation/no_ntw/{dataset}/ablation_summary.csv")
    for dataset in ["20ng", "ag_news", "dbpedia14"]
])
df_ntw = pd.concat([
    pd.read_csv(f"../results/ablation/self_aware/{dataset}/ablation_summary.csv")
    for dataset in ["20ng", "ag_news", "dbpedia14"]
])
df_ntw = df_ntw[~df_ntw["self_aware_context"]]
df = pd.concat([df_no_ntw, df_ntw])

df = df.drop([
    "num_attn_passes", "decorrelation_tau", "gamma", "self_aware_context",
    "accuracy_mean", "accuracy_std", "seed_mean", "seed_std", "ctx_len",
], axis=1)
df["dataset"] = df.dataset.replace({
    "20ng": "20NG",
    "ag_news": "AG News",
    "dbpedia14": "DBpedia14",
})
df["model_variant"] = df.model_variant.replace({
    "full": "AARTM",
    "no_nwt": "AARTM-no-N",
})
df = df.astype({
    col: float for col in df.columns if col not in ["dataset", "model_variant"]
})
df = df.reset_index(drop=True)

df

In [ ]:
datasets = ["20NG", "AG News", "DBpedia14"]
models = ["AARTM", "AARTM-no-N"]
metrics = [
    ("macro_f1_mean", "macro_f1_std", "Macro-F1"),
    ("topic_diversity_25_mean", "topic_diversity_25_std", "TD@25"),
    ("npmi_10_mean", "npmi_10_std", "NPMI@10"),
    ("c_v_10_mean", "c_v_10_std", "C_v@10"),
]

colors = {
    "AARTM": "#4C72B0",
    "AARTM-no-N": "#DD8452",
}

sns.set_theme(style="whitegrid", context="paper", font_scale=1.8)

fig, axes = plt.subplots(2, 2, figsize=(8, 6), constrained_layout=True, sharex=True)
axes = np.ravel(axes)

x = np.arange(len(datasets))
width = 0.36

for ax, (metric, std_col, title) in zip(axes, metrics):
    for j, model in enumerate(models):
        means = []
        stds = []

        for dataset in datasets:
            row = df[(df["dataset"] == dataset) & (df["model_variant"] == model)].iloc[0]
            means.append(row[metric])
            stds.append(row[std_col])

        offset = (j - 0.5) * width

        ax.bar(
            x + offset,
            means,
            width=width,
            yerr=stds,
            capsize=3,
            label=model,
            color=colors[model],
            edgecolor="black",
            linewidth=0.6,
            alpha=0.9,
        )

    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=30, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.35)

    if metric == "NPMI@10":
        ax.axhline(0, color="black", linewidth=0.7)
        ax.set_ylim(-0.04, 0.32)
    elif metric == "TD@25":
        ax.set_ylim(0.55, 0.86)
    elif metric in ["Accuracy", "Macro-F1"]:
        ax.set_ylim(0.55, 0.96)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, -0.12),
)

plt.savefig("images/table6_ntw_ablation.pdf", bbox_inches="tight")
plt.savefig("images/table6_ntw_ablation.png", bbox_inches="tight", dpi=300)
plt.show()

# Length robustness

In [ ]:
df = pd.concat([
    pd.read_csv(f"../results/length_robustness/{dataset}/length_robustness_summary.csv")
    for dataset in ["20ng", "ag_news", "dbpedia14"]
])
df.columns = [
    "dataset", "model", "tokens",
    "accuracy_mean", "accuracy_std",
    "f1_mean", "f1_std",
    "seed_mean", "seed_std",
]
df = df.drop(["accuracy_mean", "accuracy_std", "seed_mean", "seed_std"], axis=1)
df = df[~df.dataset.isna() & ~df.f1_mean.isna()].reset_index(drop=True)
df["dataset"] = df.dataset.replace({
    "20ng": "20NG",
    "ag_news": "AG News",
    "dbpedia14": "DBpedia14",
})
df["model"] = df.model.replace({
    "AttentiveTopicModel": "AARTM",
    "AttentiveTopicModelNoNWT": "AARTM-no-N",
})
df = df.astype({
    "tokens": int,
    "f1_mean": float,
    "f1_std": float,
})
df

In [ ]:
plot_df = df[df["tokens"] != 1]

datasets = ["20NG", "AG News", "DBpedia14"]
models = ["AARTM", "LDA", "NMF"]

palette = {
    "AARTM": "#1f77b4",
    "AARTM-no-N": "#ff7f0e",
    "LDA": "#2ca02c",
    "NMF": "#d62728",
}

markers = {
    "AARTM": "o",
    "AARTM-no-N": "s",
    "LDA": "^",
    "NMF": "D",
}

sns.set_theme(style="whitegrid", context="paper", font_scale=1.8)

fig, axes = plt.subplots(
    len(datasets),
    1,
    figsize=(5, 10),
    sharex=True,
)

for ax, dataset in zip(axes, datasets):
    sub = plot_df[plot_df["dataset"] == dataset]

    for model in models:
        g = sub[sub["model"] == model].sort_values("tokens")

        x = g["tokens"].values
        y = g["f1_mean"].values
        y_std = g["f1_std"].values

        ax.plot(
            x,
            y,
            label=model,
            color=palette[model],
            marker=markers[model],
            linewidth=2,
            markersize=5,
        )

        ax.fill_between(
            x,
            y - y_std,
            y + y_std,
            color=palette[model],
            alpha=0.15,
            linewidth=0,
        )

    ax.set_title(dataset)
    ax.set_xscale("log", base=2)
    ax.set_xticks(sorted(plot_df["tokens"].unique()))
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
axes[-1].set_xlabel("Observed tokens")

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower right",
    bbox_to_anchor=(0.9, 0.1),
    # ncol=4,
    # frameon=False,
)

fig.tight_layout()

plt.savefig("images/table7_length_robustness.pdf", bbox_inches="tight")
plt.savefig("images/table7_length_robustness.png", bbox_inches="tight", dpi=300)
plt.show()


# Boundary detection

In [ ]:
df = pd.concat([
    pd.read_csv(f"../results/boundary_detection/{dataset}/boundary_detection_summary.csv")
    for dataset in ["20ng", "ag_news", "dbpedia14"]
])
df.columns = [
    "dataset", "model",
    "boundary_mae_mean", "boundary_mae_std",
    "n_eval_docs_mean", "n_eval_docs_std",
    "boundary_hit@5_mean", "boundary_hit@5_std",
    "boundary_hit@10_mean", "boundary_hit@10_std",
    "seed_mean", "seed_std",
]
df = df.drop(["n_eval_docs_mean", "n_eval_docs_std", "seed_mean", "seed_std"], axis=1)
df["dataset"] = df.dataset.replace({
    "20ng": "20NG",
    "ag_news": "AG News",
    "dbpedia14": "DBpedia14",
})
df["model"] = df.model.replace({
    "AttentiveTopicModel": "AARTM",
    "AttentiveTopicModelNoNWT": "AARTM-no-N",
})

df = df.astype({
    col: float for col in df.columns if col not in ["dataset", "model"]
})
df

In [ ]:
datasets = ["20NG", "AG News", "DBpedia14"]
models = ["AARTM", "LDA", "NMF"]
metrics = [
    ("boundary_mae_mean", "boundary_mae_std", "MAE"),
    # ("boundary_hit@5_mean", "boundary_hit@5_std", "Hit@5"),
    ("boundary_hit@10_mean", "boundary_hit@10_std", "Hit@10"),
]

colors = {
    "AARTM": "#4C72B0",
    "AARTM-no-N": "#55A868",
    "LDA": "#C44E52",
    "NMF": "#8172B3",
}

sns.set_theme(style="whitegrid", context="paper", font_scale=1.8)

fig, axes = plt.subplots(
    nrows=len(metrics),
    ncols=len(datasets),
    figsize=(8.0, 5.2),
    sharex=True,
)

bar_width = 0.75
x = np.arange(len(models))

for row, metric in enumerate(metrics):
    for col, dataset in enumerate(datasets):
        ax = axes[row, col]

        subset = (
            df[df["dataset"] == dataset]
            .set_index("model")
            .loc[models]
            .reset_index()
        )

        means = subset[metric[0]].values
        stds = subset[metric[1]].values

        ax.bar(
            x,
            means,
            yerr=stds,
            width=bar_width,
            color=[colors[m] for m in models],
            edgecolor="black",
            linewidth=0.5,
            capsize=2.5,
            error_kw={"elinewidth": 0.8, "capthick": 0.8},
        )

        if row == 0:
            ax.set_title(dataset)

        if col == 0:
            ax.set_ylabel(metric[2])

        if row == len(metrics) - 1:
            ax.set_xticks(x)
            ax.set_xticklabels(models, rotation=30, ha="right")
        else:
            ax.set_xticks(x)
            ax.set_xticklabels([])

        if metric[2] == "MAE":
            ax.set_ylim(0, 10)
        else:
            ax.set_ylim(0, 1.05)

        ax.grid(axis="y", linestyle="--", alpha=0.35)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

handles = [
    plt.Rectangle((0, 0), 1, 1, color=colors[m], ec="black", lw=0.5)
    for m in models
]

fig.legend(
    handles,
    models,
    loc="lower center",
    ncol=len(models),
    frameon=False,
    bbox_to_anchor=(0.56, -0.06),
)

fig.tight_layout()

plt.savefig("images/table8_boundary_detection.pdf", bbox_inches="tight")
plt.savefig("images/table8_boundary_detection.png", bbox_inches="tight", dpi=300)
plt.show()

# Benchmark

In [ ]:
from matplotlib.cm import viridis


fig, ax = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)

vmin = benchmark_df["steady_mean_sec"].min()
vmax = benchmark_df["steady_mean_sec"].max()

cmap = viridis.copy()
cmap.set_bad(color="lightgray")

for i, model_type in enumerate(["ContextTopicModel", "AARTM"]):
    for j, batch_size in enumerate([None, 10_000, 100_000]):
        model_mask = benchmark_df.model == model_type
        if batch_size is None:
            batch_mask = (
                (benchmark_df.batched == False)
                & (benchmark_df.batch_size == 10_000)
            )
        else:
            batch_mask = (
                (benchmark_df.batch_size == batch_size)
                & benchmark_df.batched
            )
        records = benchmark_df[model_mask & batch_mask].copy()

        mean_pivot = records.pivot(
            index="ctx_len",
            columns="n_topics",
            values="steady_mean_sec",
        )
        std_pivot = records.pivot(
            index="ctx_len",
            columns="n_topics",
            values="steady_std",
        )
        if 100 not in mean_pivot.index:
            mean_pivot.loc[100] = [np.nan, np.nan, np.nan, np.nan]
            std_pivot.loc[100] = [np.nan, np.nan, np.nan, np.nan]

        def prepare_label(m, s):
            if np.isnan(m):
                return "NaN"
            return f"{m:.2f}s\n±{s:.2f}"

        labels = np.vectorize(prepare_label)(
            mean_pivot.values,
            std_pivot.reindex_like(mean_pivot).values
        )
        sns.heatmap(
            mean_pivot,
            mask=mean_pivot.isna(),
            cmap=cmap,
            annot=labels,
            fmt="",
            vmin=vmin,
            vmax=vmax,
            cbar=False,
            ax=ax[i][j],
        )

        batch_title = "non-batched" if batch_size is None else f"batch={batch_size:,}"
        ax[i][j].set_title(batch_title)
        ax[i][j].set_xlabel(None)

        if j == 0:
            ax[i][j].set_ylabel(model_type)
        else:
            ax[i][j].set_ylabel(None)
        ax[i][j].tick_params(axis="y", labelrotation=0)

fig.supxlabel("Number of topics")
fig.supylabel("Context size")
fig.tight_layout()
plt.show()